# Amazon Bedrock Video Generation

This notebook demonstrates video generation using Amazon Nova Reel, AWS's foundation model for creating videos from text prompts. Video generation represents a significant advancement in generative AI, enabling creation of dynamic visual content from simple text descriptions.

## Key Concepts:
- **Text-to-Video**: Generate videos directly from text descriptions
- **Asynchronous Processing**: Video generation requires longer processing times
- **S3 Integration**: Generated videos are stored in S3 buckets
- **Job Management**: Track generation progress and retrieve results

## Use Cases:
- **Marketing Content**: Create promotional videos and advertisements
- **Educational Materials**: Generate instructional and explanatory videos
- **Social Media**: Produce engaging content for platforms
- **Prototyping**: Quickly visualize concepts and ideas
- **Entertainment**: Create short-form creative content

## Asynchronous Processing Model

Video generation is computationally intensive and requires asynchronous processing:

### Why Asynchronous?
- **Processing Time**: Video generation can take several minutes
- **Resource Management**: Prevents API timeouts and connection issues
- **Scalability**: Allows handling multiple concurrent requests
- **Cost Efficiency**: Optimizes resource utilization

### Workflow Pattern:
1. **Submit Job**: Send generation request with parameters
2. **Receive Job ID**: Get unique identifier for tracking
3. **Poll Status**: Check job progress periodically
4. **Retrieve Results**: Download completed video from S3

### S3 Storage Requirements:
- Generated videos are automatically stored in specified S3 bucket
- Bucket must have appropriate permissions for Bedrock service
- Consider lifecycle policies for cost management

In [1]:
import boto3
import json
import random

# Initialize clients for video generation and S3 storage
bedrock_runtime = boto3.client("bedrock-runtime", region_name="us-east-1")
bedrock = boto3.client(service_name="bedrock", region_name="us-east-1")  
s3 = boto3.client("s3")

# Amazon Nova Reel - specialized model for video generation
model_id = "amazon.nova-reel-v1:0"

# Text prompt describing the desired video content
prompt = "A person dancing on a mountain."

# Generate random seed for reproducible results (optional)
# Same seed with same prompt will produce similar videos
seed = random.randint(0, 2147483646)

# Configure video generation parameters
model_input = {
    "taskType": "TEXT_VIDEO",  # Specify text-to-video generation
    "textToVideoParams": {"text": prompt},
    "videoGenerationConfig": {
        "fps": 24,              # Frames per second (standard for smooth video)
        "durationSeconds": 6,   # Video length (longer = more expensive)
        "dimension": "1280x720", # HD resolution (720p)
        "seed": seed,           # For reproducible generation
    },
}

# Configure S3 output location
# Ensure bucket exists and has proper permissions
output_config = {
    "s3OutputDataConfig": {
        "s3Uri": "s3://gen-ai-exercise-ph/video/"  # Replace with your bucket
    }
}

# Submit asynchronous video generation job
response = bedrock_runtime.start_async_invoke(
    modelId=model_id,
    modelInput=model_input,
    outputDataConfig=output_config,
)

# Store invocation ARN for status checking
invocation_arn = response["invocationArn"]
print("✅ Job submitted!")
print("Invocation ARN:", invocation_arn)

✅ Job submitted!
Invocation ARN: arn:aws:bedrock:us-east-1:206204551974:async-invoke/uzhowoivsstg


## Video Generation Parameters

Understanding the configuration options helps optimize results and costs:

### Video Quality Settings:
- **fps (Frames Per Second)**:
  - 24 fps: Standard cinematic quality
  - 30 fps: Smooth motion, higher cost
  - Lower fps: Reduced cost but choppier motion

- **dimension (Resolution)**:
  - "1280x720" (720p HD): Good quality, reasonable cost
  - "1920x1080" (1080p Full HD): Higher quality, increased cost
  - "640x480" (SD): Lower cost, reduced quality

### Duration Considerations:
- **Short videos (2-6 seconds)**: Ideal for social media, lower cost
- **Medium videos (6-15 seconds)**: Good for demonstrations
- **Longer videos**: Exponentially more expensive

### Seed Parameter:
- **Reproducibility**: Same seed + prompt = similar results
- **Variation**: Different seeds create diverse interpretations
- **Testing**: Use fixed seeds during development, random in production

In [2]:
# Check the status of the video generation job
# This should be called periodically until completion
job_status = bedrock_runtime.get_async_invoke(invocationArn=invocation_arn)
print("Current Status:", job_status["status"])

Current Status: InProgress


## Job Status Monitoring

Video generation jobs progress through several states:

### Status Values:
- **InProgress**: Job is currently being processed
- **Completed**: Video generation finished successfully
- **Failed**: Job encountered an error
- **Stopped**: Job was manually cancelled

### Monitoring Best Practices:
```python
import time

def wait_for_completion(invocation_arn, max_wait_time=600):
    start_time = time.time()
    
    while time.time() - start_time < max_wait_time:
        status = bedrock_runtime.get_async_invoke(invocationArn=invocation_arn)
        
        if status["status"] == "Completed":
            return status
        elif status["status"] == "Failed":
            raise Exception(f"Job failed: {status.get('failureMessage', 'Unknown error')}")
        
        time.sleep(30)  # Check every 30 seconds
    
    raise TimeoutError("Job did not complete within expected time")
```

### Typical Processing Times:
- **2-6 second videos**: 2-5 minutes
- **6-15 second videos**: 5-15 minutes
- **Complex scenes**: May take longer
- **High resolution**: Increases processing time

## Result Retrieval and Management

Once the job completes, the video is available in your S3 bucket:

### Accessing Generated Videos:
```python
# When job status is "Completed"
completed_job = bedrock_runtime.get_async_invoke(invocationArn=invocation_arn)

# Extract S3 location from response
output_location = completed_job['outputDataConfig']['s3OutputDataConfig']['s3Uri']

# Download video file
s3.download_file(
    Bucket='your-bucket-name',
    Key='video/generated-video.mp4',
    Filename='local-video.mp4'
)
```

### File Management:
- **Naming Convention**: Videos are automatically named with unique identifiers
- **Format**: Generated videos are typically in MP4 format
- **Size**: File size depends on duration, resolution, and complexity
- **Cleanup**: Implement lifecycle policies to manage storage costs

### Integration Patterns:
- **Web Applications**: Stream videos directly from S3 with CloudFront
- **Mobile Apps**: Download and cache videos locally
- **Batch Processing**: Generate multiple videos and process in parallel
- **Content Pipelines**: Integrate with video editing and distribution workflows

## Production Implementation Considerations

### Cost Optimization:
- **Resolution**: Use appropriate resolution for intended use
- **Duration**: Keep videos as short as possible for the use case
- **Batch Processing**: Group similar requests to optimize resource usage
- **Caching**: Store and reuse videos for similar prompts

### Quality Control:
- **Prompt Engineering**: Craft detailed, specific prompts for better results
- **Seed Management**: Use seeds for consistent branding or style
- **Review Process**: Implement human review for sensitive content
- **Fallback Options**: Have backup content for failed generations

### Scalability:
- **Queue Management**: Implement job queues for high-volume requests
- **Rate Limiting**: Respect API limits and implement backoff strategies
- **Monitoring**: Track job success rates and processing times
- **Error Handling**: Robust retry logic for failed jobs

### Security and Compliance:
- **S3 Permissions**: Secure bucket access with appropriate IAM policies
- **Content Filtering**: Use guardrails to prevent inappropriate content
- **Data Retention**: Implement policies for video storage and deletion
- **Audit Logging**: Track video generation requests and usage

## Prompt Engineering for Video Generation

Effective prompts are crucial for high-quality video generation:

### Best Practices:
- **Be Specific**: "A person dancing salsa on a mountain peak at sunset" vs "A person dancing"
- **Include Context**: Describe setting, lighting, mood, and style
- **Specify Motion**: Describe the type and speed of movement desired
- **Consider Composition**: Mention camera angles, framing, and perspective

### Example Prompts:
```
Good: "A professional chef preparing pasta in a modern kitchen, close-up shots of hands kneading dough, warm lighting"

Better: "A professional chef in white uniform preparing fresh pasta in a modern stainless steel kitchen, close-up shots of experienced hands kneading golden dough on marble counter, warm ambient lighting, steam rising from boiling water"
```

### Common Pitfalls:
- **Vague descriptions**: Lead to unpredictable results
- **Conflicting elements**: May confuse the generation process
- **Overly complex scenes**: Can result in lower quality output
- **Inappropriate content**: Will be blocked by content filters

### Testing Strategy:
1. Start with simple, clear prompts
2. Gradually add detail and complexity
3. Use consistent seeds to test prompt variations
4. Build a library of successful prompt patterns